In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np


In [2]:
def parse_duration_to_seconds(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s == "":
        return np.nan
    parts = s.split(":")
    try:
        parts = [float(p) for p in parts]
    except Exception:
        # maybe already numeric string (seconds)
        try:
            return float(s)
        except Exception:
            return np.nan
    # seconds only
    if len(parts) == 1:
        return parts[0]
    # mm:ss(.fraction)
    if len(parts) == 2:
        m, sec = parts
        return m * 60.0 + sec
    # hh:mm:ss
    if len(parts) == 3:
        h, m, sec = parts
        return h * 3600.0 + m * 60.0 + sec
    return np.nan

In [5]:
def parse_date(v):
    if pd.isna(v):
        return None
    try:
        return pd.to_datetime(v, utc=True, errors="coerce")
    except Exception:
        return pd.to_datetime(v, errors="coerce")
def compute_age_months(dob, vid_date):
    if dob is None or pd.isna(dob) or vid_date is None or pd.isna(vid_date):
        return np.nan
    dob_ts = pd.to_datetime(dob, utc=True, errors="coerce")
    vd_ts = pd.to_datetime(vid_date, utc=True, errors="coerce")
    if pd.isna(dob_ts) or pd.isna(vd_ts):
        return np.nan
    # use total seconds to avoid .days/.seconds mix and convert to months (~30.4375 days)
    seconds = (vd_ts - dob_ts).total_seconds()
    months = seconds / (30.4375 * 86400.0)
    return float(months)
def session_from_age_months(m):
    if pd.isna(m):
        return "session_unknown"
    m = float(m)
    if 12 <= m <= 16:
        return "12-16 months"
    if 34 <= m <= 38:
        return "34-38 months"
    # older mapping used in checkpoints: 14_month etc. keep human labels by default
    if 10 <= m < 24:
        return "12-16 months"
    if 30 <= m < 40:
        return "34-38 months"
    return "session_unknown"

In [ ]:

out_dir = "."
p = Path("/orcd/data/satra/002/datasets/SAILS/data4analysis/Video Rating Data/SAILS_RATINGS_ALL_DEDUPLICATED_NotForFinalAnalyses_2025.10.csv")
df = pd.read_csv(p, dtype=str)

# normalize common column name variants
df.columns = [c.strip() for c in df.columns]
def col_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

src_col = col_first(df, ["SourceFile", "sourcefile", "Source_File"])
id_col = col_first(df, ["ID", "Id", "id"])
fn_col = col_first(df, ["FileName", "FileName", "filename", "File_Name"])
dur_col = col_first(df, ["Vid_duration", "Vid_Duration", "duration"])
dob_col = col_first(df, ["DOB", "Dob", "dob"])
vd_col = col_first(df, ["VideoDate", "VideoDate", "Video Date", "Video_Date"])
age_col = col_first(df, ["Age", "age", "age_in_months", "Age_in_months"])
timepoint_col = col_first(df, ["timepoint", "Timepoint", "TimePoint"])
date_valid_col = col_first(df, ["DateValidityScore", "Date Validity Score", "Date_Validity_Score"])

required = [src_col, id_col, fn_col]
if not all(required):
    raise SystemExit("CSV missing one of required columns: SourceFile, ID, FileName")

out = pd.DataFrame()
out["sourcefile"] = df[src_col].astype(str).str.strip()
out["video_filename"] = df[fn_col].astype(str).apply(lambda s: Path(s).name)
out["participant_id"] = df[id_col].astype(str).str.strip().str.upper()

# duration parse
if dur_col:
    out["duration"] = df[dur_col].apply(parse_duration_to_seconds)
else:
    out["duration"] = pd.NA

# video_date parse
if vd_col:
    out["video_date"] = df[vd_col].apply(parse_date).dt.tz_convert(None).dt.strftime("%Y-%m-%d")
else:
    out["video_date"] = pd.NA

out["date_validity_score"] = df[date_valid_col] if date_valid_col else pd.NA

ages = []
for i, row in df.iterrows():
    dob = row.get(dob_col) if dob_col else None
    vd = row.get(vd_col) if vd_col else None
    age_m = compute_age_months(dob, vd) if dob or vd else np.nan
    if np.isnan(age_m):
        raw_age = row.get(age_col) if age_col else None
        if raw_age is None or (isinstance(raw_age, float) and np.isnan(raw_age)) or str(raw_age).strip()=="":
            age_m = np.nan
        else:
            try:
                a = float(str(raw_age))
                # heuristic: if a looks like years (<=6) convert to months
                if a <= 6:
                    age_m = a * 12.0
                else:
                    # assume already months
                    age_m = a
            except Exception:
                age_m = np.nan
    ages.append(age_m)
out["age"] = ages

if timepoint_col and timepoint_col in df.columns:
    out["session_id"] = df[timepoint_col].fillna("").astype(str).replace("", "session_unknown")
else:
    out["session_id"] = out["age"].apply(lambda m: session_from_age_months(m))

out = out.drop_duplicates(subset=["participant_id", "video_filename"])
bids_root = Path(out_dir) / "bids-dataset"
bids_root.mkdir(parents=True, exist_ok=True)

participants_cols = ["participant_id", "session_id", "video_filename", "duration", "age", "video_date", "date_validity_score", "sourcefile"]
out.to_csv(bids_root / "participants.tsv", sep="\t", index=False, columns=participants_cols)

participants_json = {
    "participant_id": {"Description": "Original alphanumeric participant identifier (without 'sub-')."},
    "session_id": {"Description": "Session label (e.g., '12-16 months', '34-38 months')."},
    "video_filename": {"Description": "Source video file name (basename)."},
    "duration": {"Description": "Video duration (seconds).", "Units": "s"},
    "age": {"Description": "Age at video time (months).", "Units": "months"},
    "video_date": {"Description": "Date of the video (ISO YYYY-MM-DD)."},
    "date_validity_score": {"Description": "Date validity score from SAILS ratings CSV."},
    "sourcefile": {"Description": "Original full source file path from SAILS CSV."},
}
import json
with open(bids_root / "participants.json", "w") as f:
    json.dump(participants_json, f, indent=4)

print(f"[OK] wrote: {bids_root / 'participants.tsv'}")



[OK] wrote: bids-dataset/participants.tsv
